# <font color="#418FDE" size="6.5" uppercase>**Regression Clustering**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Evaluate regression models using scale-dependent and scale-independent metrics. 
- Evaluate clustering results using internal and external metrics. 
- Use dummy classifiers and regressors as baseline comparisons. 


## **1. Regression Metrics**

### **1.1. Explained Variance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_01_01.jpg?v=1784038513" width="250">



>* Measures how well predictions capture variation
>* Shows whether data patterns are represented

>* Unitless metric for comparing different scales
>* Shows variation captured, not error size

>* High variance explained can still hide errors
>* Compare with baselines and other metrics



In [ ]:
#@title Python Code - Explained Variance

# This example demonstrates explained variance for regression.
# It compares captured variation with error size.
# The output shows why metrics complement each other.

import numpy as np
import sklearn
from sklearn.metrics import explained_variance_score
from sklearn.metrics import mean_absolute_error

# These actual values represent electricity demand in megawatt-hours.
actual_demand = np.array([100, 120, 140, 160, 180, 200], dtype=float)

# This model follows the ups and downs but stays high.
shifted_predictions = actual_demand + 30

# This model has similar average error but misses variation.
flat_predictions = np.array([150, 150, 150, 150, 150, 150], dtype=float)

# Validate that both prediction arrays match the target shape.
if shifted_predictions.shape != actual_demand.shape:
    raise ValueError("Shifted predictions must match actual values.")

if flat_predictions.shape != actual_demand.shape:
    raise ValueError("Flat predictions must match actual values.")

# Explained variance rewards following the target's movement.
shifted_ev = explained_variance_score(actual_demand, shifted_predictions)
flat_ev = explained_variance_score(actual_demand, flat_predictions)

# Mean absolute error shows the typical error size.
shifted_mae = mean_absolute_error(actual_demand, shifted_predictions)
flat_mae = mean_absolute_error(actual_demand, flat_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Shifted model explained variance: {shifted_ev:.2f}")
print(f"Flat model explained variance: {flat_ev:.2f}")
print(f"Shifted model MAE: {shifted_mae:.1f} MWh")
print(f"Flat model MAE: {flat_mae:.1f} MWh")
print("Explained variance is unitless; MAE keeps the target units.")



### **1.2. Absolute Error Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_01_02.jpg?v=1784038515" width="250">



>* Measure prediction errors in original target units
>* Show average error size, ignoring direction

>* Errors depend on the target’s scale
>* Judge errors using real-world context

>* Less affected by outliers than squared errors
>* Use with metrics when big errors matter



In [ ]:
#@title Python Code - Absolute Error Metrics

# This example measures absolute prediction errors.
# Mean absolute error keeps original units.
# The output compares model and baseline errors.

import numpy as np
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Small in-memory data keeps the lesson focused.
house_size_sq_m = np.array([45, 55, 62, 70, 85, 95, 110, 125, 140, 160])
house_price_thousands = np.array([180, 205, 230, 250, 300, 330, 370, 410, 455, 520])

# Scikit-learn expects features in a two-dimensional table.
features = house_size_sq_m.reshape(-1, 1)
target = house_price_thousands

# Validate that each house has one matching price.
if len(features) != len(target):
    raise ValueError("Each house must have one matching price.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=42
)

# Fit one simple regression model.
model = LinearRegression()
model.fit(X_train, y_train)

# Fit a baseline that always predicts the training mean.
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

# Predict prices for the held-out test houses.
model_predictions = model.predict(X_test)
baseline_predictions = baseline.predict(X_test)

# MAE is the average absolute error in target units.
model_mae = mean_absolute_error(y_test, model_predictions)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Linear regression MAE: {model_mae:.1f} thousand dollars")
print(f"Mean baseline MAE: {baseline_mae:.1f} thousand dollars")
print(f"First test home actual: {y_test[0]:.0f} thousand dollars")
print(f"First test home prediction: {model_predictions[0]:.0f} thousand dollars")



### **1.3. Percentage Log Errors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_01_03.jpg?v=1784038517" width="250">



>* Measure errors relative to actual values
>* Compare performance across different target scales

>* Percentage errors are easy to interpret
>* Beware zeros, tiny values, and business costs

>* Use logs for proportional, skewed targets
>* Handle zeros, negatives, and interpretation carefully



In [ ]:
#@title Python Code - Percentage Log Errors

# Compare percentage and logarithmic regression errors.
# Relative metrics reveal scale independent mistakes.
# Printed results show different metric behavior.

import numpy as np
import sklearn
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_log_error

# True values stay positive for percentage and log metrics.
y_true = np.array([10, 20, 100, 200, 1000, 2000], dtype=float)
y_pred = np.array([12, 18, 120, 180, 1200, 1800], dtype=float)

# Validate the key assumption before computing these metrics.
if np.any(y_true <= 0) or np.any(y_pred < 0):
    raise ValueError("Use positive true values and nonnegative predictions.")

# Absolute errors grow with the size of the target.
absolute_errors = np.abs(y_true - y_pred)
percentage_errors = absolute_errors / y_true * 100

# MAPE summarizes average proportional error as a percentage.
mape_percent = mean_absolute_percentage_error(y_true, y_pred) * 100
rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"True values: {y_true.astype(int).tolist()}")
print(f"Predictions: {y_pred.astype(int).tolist()}")
print(f"Absolute errors: {absolute_errors.astype(int).tolist()}")
print(f"Percentage errors: {np.round(percentage_errors, 1).tolist()}%")
print(f"MAPE: {mape_percent:.1f}%")
print(f"RMSLE: {rmsle:.3f}")



## **2. Clustering Metrics**

### **2.1. Median Quantile Deviance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_02_01.jpg?v=1784038500" width="250">



>* Measures cluster fit using typical members
>* Reduces distortion from extreme observations

>* Checks typical cluster tightness robustly
>* Separates outlier effects from true inconsistency

>* Lower deviance means tighter cluster fit
>* Combine with other validation methods



In [ ]:
#@title Python Code - Median Quantile Deviance

# This example measures robust clustering compactness.
# Median deviance reduces outlier influence.
# Lower values indicate tighter typical clusters.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Create compact clusters plus a few unusual observations.
features, true_groups = make_blobs(
    n_samples=120, centers=3, cluster_std=0.65, random_state=42
)

# Add deterministic outliers to challenge mean-based evaluation.
outliers = np.array([[7.5, 7.0], [-7.0, 6.5], [6.5, -6.5]])
features = np.vstack([features, outliers])

# Fit one simple clustering model for the demonstration.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features)

# Validate that every cluster received at least one observation.
cluster_sizes = np.bincount(cluster_labels, minlength=3)
if np.any(cluster_sizes == 0):
    raise ValueError("Each cluster must contain at least one observation.")

# Compute distances from each point to its assigned cluster center.
assigned_centers = kmeans.cluster_centers_[cluster_labels]
distances = np.linalg.norm(features - assigned_centers, axis=1)

# Compare average deviance with median quantile deviance.
mean_deviance = np.mean(distances)
median_quantile_deviance = np.median(distances)
upper_quartile_deviance = np.quantile(distances, 0.75)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Mean distance to assigned center: {mean_deviance:.2f}")
print(f"Median quantile deviance: {median_quantile_deviance:.2f}")
print(f"75th percentile distance: {upper_quartile_deviance:.2f}")

# Plot distances so the median-based summary is visible.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(distances, bins=14, color="steelblue", edgecolor="white")
ax.axvline(median_quantile_deviance, color="darkorange", linewidth=3)
ax.axvline(mean_deviance, color="black", linestyle="--", linewidth=2)

ax.set_title("Distances Within Assigned Clusters")
ax.set_xlabel("Distance to assigned cluster center")
ax.set_ylabel("Number of observations")
ax.legend(["Median quantile deviance", "Mean distance"])
plt.show()



### **2.2. External Cluster Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_02_02.jpg?v=1784038503" width="250">



>* Compare clusters with trusted outside labels
>* Check alignment with meaningful real-world categories

>* Compare clusters with labels, ignoring names
>* Use ARI, NMI, homogeneity, and completeness

>* Reference labels may be imperfect or misleading
>* Combine external scores with broader evaluation



In [ ]:
#@title Python Code - External Cluster Validation

# This example compares clusters with trusted labels.
# External metrics ignore arbitrary cluster number names.
# Higher scores show stronger label agreement.

import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import homogeneity_completeness_v_measure
from sklearn.preprocessing import StandardScaler

# Load a small dataset with known species labels.
iris = load_iris()
features = iris.data
true_labels = iris.target

# Validate the basic shape before clustering.
if features.shape[0] != true_labels.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Scale features so measurements contribute more fairly.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Cluster without giving KMeans the species labels.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_features)

# Compute external validation metrics after clustering.
ari = adjusted_rand_score(true_labels, cluster_labels)
nmi = normalized_mutual_info_score(true_labels, cluster_labels)
homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(
    true_labels,
    cluster_labels,
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Adjusted Rand index: {ari:.3f}")
print(f"Normalized mutual information: {nmi:.3f}")
print(f"Homogeneity, completeness, V-measure: {homogeneity:.3f}, {completeness:.3f}, {v_measure:.3f}")
print("These scores compare clusters with labels KMeans never saw.")



### **2.3. Internal Cluster Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_02_03.jpg?v=1784038505" width="250">



>* Assess clusters without external labels
>* Compare compactness and separation cautiously

>* Silhouette, Davies-Bouldin, Calinski-Harabasz compare cluster quality
>* Different metrics may prefer different cluster solutions

>* Know metric assumptions and data sensitivities
>* Combine scores with context and interpretability



In [ ]:
#@title Python Code - Internal Cluster Metrics

# This example compares internal clustering metrics.
# Compact separated clusters should score better.
# The printed scores guide cluster selection.

import numpy as np
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Create a small dataset with three natural groups.
features, true_groups = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.75,
    random_state=42,
)

# Validate the generated data before clustering.
if features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

# Scale features because distance based metrics need comparable units.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

print("scikit-learn version:", sklearn.__version__)
print("k  silhouette  davies_bouldin  calinski_harabasz")

# Compare several possible cluster counts using internal metrics.
for cluster_count in [2, 3, 4, 5]:
    model = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
    labels = model.fit_predict(scaled_features)
    silhouette = silhouette_score(scaled_features, labels)
    davies = davies_bouldin_score(scaled_features, labels)
    calinski = calinski_harabasz_score(scaled_features, labels)
    print(cluster_count, round(silhouette, 3), round(davies, 3), round(calinski, 1))

print("Higher silhouette and Calinski, lower Davies-Bouldin, are preferred.")



## **3. Baseline Comparisons**

### **3.1. Dummy Classifiers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_03_01.jpg?v=1784038509" width="250">



>* Simple rules set classification baselines
>* Compare models against trivial performance

>* Expose misleading accuracy in imbalanced datasets
>* Compare models against simple prediction strategies

>* Compare baselines with the same metrics
>* Require models to beat simple baselines



In [ ]:
#@title Python Code - Dummy Classifiers

# Compare real classifiers against simple dummy baselines.
# Dummy strategies reveal misleading accuracy on imbalance.
# Results show whether learning beats trivial guessing.

import sklearn
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

# Create a small imbalanced classification dataset.
features, target = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=3,
    n_redundant=0,
    weights=[0.9, 0.1],
    random_state=42,
)

# Check that the dataset has matching rows and labels.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and labels must have matching rows.")

# Split data while preserving the class imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Fit one real model and two dummy baselines.
models = {
    "Most frequent dummy": DummyClassifier(strategy="most_frequent"),
    "Stratified dummy": DummyClassifier(strategy="stratified", random_state=42),
    "Logistic regression": LogisticRegression(max_iter=500, random_state=42),
}

# Evaluate every model with the same two metrics.
print("scikit-learn version:", sklearn.__version__)
print("Model | accuracy | balanced accuracy")
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    balanced = balanced_accuracy_score(y_test, predictions)
    print(model_name, "|", round(accuracy, 3), "|", round(balanced, 3))



### **3.2. Dummy Regressor Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_03_02.jpg?v=1784038507" width="250">



>* Simple regression baseline using fixed predictions
>* Checks whether models beat basic guesses

>* Choose mean, median, or constant baselines
>* Compare models against simple transparent predictions

>* Compare models against simple baseline predictions
>* Small gains may signal weak predictive value



In [ ]:
#@title Python Code - Dummy Regressor Baselines

# Compare real regression against simple dummy baselines.
# Dummy regressors reveal whether learning adds value.
# Printed metrics show baseline and model performance.

import numpy as np
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Load a small packaged regression dataset.
diabetes = load_diabetes()
features = diabetes.data
target = diabetes.target

# Validate the basic regression data shape.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split data before fitting any model.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Fit two transparent dummy baselines.
mean_dummy = DummyRegressor(strategy="mean")
median_dummy = DummyRegressor(strategy="median")
mean_dummy.fit(X_train, y_train)
median_dummy.fit(X_train, y_train)

# Fit one simple model for comparison.
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predict on the same held-out test set.
mean_predictions = mean_dummy.predict(X_test)
median_predictions = median_dummy.predict(X_test)
model_predictions = linear_model.predict(X_test)

# Summarize each approach with beginner-friendly metrics.
results = [
    ("Mean dummy", mean_predictions),
    ("Median dummy", median_predictions),
    ("Linear regression", model_predictions),
]

print("scikit-learn version:", sklearn.__version__)
print("Model              MAE    R2")
for name, predictions in results:
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    print(f"{name:<17} {mae:5.1f} {r2:5.2f}")

best_dummy_mae = min(
    mean_absolute_error(y_test, mean_predictions),
    mean_absolute_error(y_test, median_predictions),
)

model_mae = mean_absolute_error(y_test, model_predictions)
improvement = best_dummy_mae - model_mae
print(f"MAE improvement over best dummy: {improvement:.1f}")



### **3.3. Baseline Reporting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_29/Lecture_B/image_03_03.jpg?v=1784038511" width="250">



>* Compare model scores with simple baselines
>* Reveal imbalance or weak learned patterns

>* State the dummy strategy and test setup
>* Compare gains against added model complexity

>* Baselines make results understandable for stakeholders
>* Compare models against simple practical rules



In [ ]:
#@title Python Code - Baseline Reporting

# Compare a model against a simple baseline.
# Dummy regressors reveal whether learning adds value.
# The printed scores show fair baseline reporting.

import sklearn
from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Load a small packaged regression dataset.
diabetes = load_diabetes()
features = diabetes.data
target = diabetes.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Use the same split for both comparisons.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Fit a dummy regressor that predicts the training mean.
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

# Fit one simple real model on the same training data.
model = LinearRegression()
model.fit(X_train, y_train)
model_predictions = model.predict(X_test)

# Evaluate both methods with the same metric.
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
model_mae = mean_absolute_error(y_test, model_predictions)
improvement = baseline_mae - model_mae

print(f"scikit-learn version: {sklearn.__version__}")
print("Baseline strategy: predict the training-set mean target.")
print(f"DummyRegressor MAE: {baseline_mae:.1f} disease-score points")
print(f"LinearRegression MAE: {model_mae:.1f} disease-score points")
print(f"Improvement over baseline: {improvement:.1f} disease-score points")



# <font color="#418FDE" size="6.5" uppercase>**Regression Clustering**</font>


In this lecture, you learned to:
- Evaluate regression models using scale-dependent and scale-independent metrics. 
- Evaluate clustering results using internal and external metrics. 
- Use dummy classifiers and regressors as baseline comparisons. 

In the next Module (Module 30), we will go over 'Routing Inspection'